# Assignment 3: Milestone I Natural Language Processing
## Task 2 and Task 3
#### Student Name:

1.   Vo Ngoc Dung - S4124370
2.   Tang Hoang Ha - S4147768
3.   Nguyen Anh Duc - S4136756
4.   Nguyen Quoc Trong Nghia - S3343711

Environment: Python 3 and Jupyter notebook

Libraries used:
- **pandas, numpy** — data manipulation and numerical operations
- **scikit-learn** — TfidfVectorizer, LinearRegression, RandomForestClassifier, cross-validation, evaluation metrics
- **gensim** — pretrained FastText word embeddings (`fasttext-wiki-news-subwords-300`)
- **collections.Counter** — efficient token frequency counting

## Introduction

This notebook implements two tasks:

**Task 2 — Feature Representation:** Convert the preprocessed reviews (from Task 1) into three numeric representations suitable for machine learning:
1. Sparse count vectors (bag-of-words using `vocab.txt`)
2. Unweighted average FastText word vectors (300-d)
3. TF-IDF weighted average FastText word vectors (300-d)

**Task 3 — Classification & Regression:** Train and evaluate models to answer two questions:
- **Q1 (Regression):** Can review text predict `review_rating`? Evaluated with Linear Regression on FastText features.
- **Q2 (Classification):** Can we predict `is_a_buyer` from text + structured features? Evaluated with Random Forest using 5-fold stratified cross-validation.

All experiments use **5-fold cross-validation** to ensure robust, unbiased performance estimates.

## Importing Libraries

In [1]:
# !pip install -q --upgrade pip setuptools wheel
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import os
import gensim.downloader as gensim_api

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [3]:
PROCESSED_CSV = "processed.csv"
VOCAB = "vocab.txt"
COUNT_VECTOR = "count_vectors.txt"


def GenerateCountVector():
    # Load vocabulary (word -> integer index)
    vocab = {}
    with open(VOCAB, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            word, idx = line.rsplit(":", 1)
            vocab[word] = int(idx)

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    # review_text contains space-separated tokens produced by Task 1
    review_texts = df["review_text"].fillna("").astype(str).tolist()

    # Build and write sparse count vectors
    with open(COUNT_VECTOR, "w", encoding="utf-8") as out:
        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Count only tokens that exist in the vocabulary
            counts = Counter(token for token in tokens if token in vocab)
            # Sort by word integer index for a consistent ordering
            sparse_entries = sorted(
                (vocab[word], freq) for word, freq in counts.items()
            )
            sparse_str = ",".join(f"{idx}:{freq}" for idx, freq in sparse_entries)
            out.write(f"#{review_idx},{sparse_str}\n")

    print(f"Count vectors saved to '{COUNT_VECTOR}' ({len(review_texts)} reviews).")


# Run
GenerateCountVector()

Count vectors saved to 'count_vectors.txt' (60407 reviews).


In [ ]:
UNWEIGHTED_VECTOR = "unweighted_vectors.txt"
WEIGHTED_VECTOR = "weighted_vectors.txt"
FASTTEXT_MODEL_NAME = "fasttext-wiki-news-subwords-300"


def GenerateEmbeddingVectors():
    # Load pretrained FastText model
    print(f"Loading FastText model '{FASTTEXT_MODEL_NAME}'")
    fasttext_model = gensim_api.load(FASTTEXT_MODEL_NAME)
    vector_size = fasttext_model.vector_size
    print(f"Model loaded. Vector size: {vector_size}")

    # Load processed reviews
    df = pd.read_csv(PROCESSED_CSV)
    review_texts = df["review_text"].fillna("").astype(str).tolist()
    print("Load review")
    # Fit TF-IDF over the full corpus (for weighted representation)
    # tokenizer=str.split preserves the already-cleaned tokens from Task 1
    tfidf = TfidfVectorizer(tokenizer=str.split, lowercase=False, token_pattern=None)
    tfidf_matrix = tfidf.fit_transform(review_texts)
    tfidf_feature_names = tfidf.get_feature_names_out()
    tfidf_vocab = {word: idx for idx, word in enumerate(tfidf_feature_names)}

    # Generate and write vectors
    with open(UNWEIGHTED_VECTOR, "w", encoding="utf-8") as uw_out, open(
        WEIGHTED_VECTOR, "w", encoding="utf-8"
    ) as w_out:

        for review_idx, text in enumerate(review_texts):
            tokens = text.split()
            # Keep only tokens the FastText model knows
            valid_tokens = [t for t in tokens if t in fasttext_model]

            if valid_tokens:
                vectors = np.array([fasttext_model[t] for t in valid_tokens])

                # Unweighted: simple average of word vectors
                unweighted_vec = vectors.mean(axis=0)

                # Weighted: TF-IDF weighted average
                tfidf_row = tfidf_matrix[review_idx]
                weights = np.array(
                    [
                        tfidf_row[0, tfidf_vocab[t]] if t in tfidf_vocab else 0.0
                        for t in valid_tokens
                    ]
                )
                weight_sum = weights.sum()
                if weight_sum > 0:
                    weighted_vec = (vectors * weights[:, np.newaxis]).sum(
                        axis=0
                    ) / weight_sum
                else:
                    weighted_vec = unweighted_vec
            else:
                unweighted_vec = np.zeros(vector_size)
                weighted_vec = np.zeros(vector_size)

            uw_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in unweighted_vec) + "\n"
            )
            w_out.write(
                f"#{review_idx}," + ",".join(f"{v:.6f}" for v in weighted_vec) + "\n"
            )

    print(f"Number of reviews: {len(review_texts)}")
    print(f"Unweighted vectors saved to '{UNWEIGHTED_VECTOR}'")
    print(f"Weighted vectors saved to '{WEIGHTED_VECTOR}'")


# Run
GenerateEmbeddingVectors()

Loading FastText model 'fasttext-wiki-news-subwords-300'
